# Movie rating classification

## main purpose
Our main purpose  here given a set of movies and set of users rating the movies we want find model that
predict given a movie and user what the user will rate this movie

## data set
first we will install the data set

In [3]:
%pip install kagglehub
import kagglehub

# Download latest version
path = kagglehub.dataset_download("grouplens/movielens-20m-dataset")

print("Path to dataset files:", path)

Note: you may need to restart the kernel to use updated packages.
Path to dataset files: /root/.cache/kagglehub/datasets/grouplens/movielens-20m-dataset/versions/1


In [4]:
from matplotlib import pylab
#from google.colab import drive

import matplotlib.pyplot as plt
import pandas as pd
import sys
!{sys.executable} -m pip install scikit-learn
!{sys.executable} -m pip install seaborn
from sklearn.model_selection import train_test_split
import numpy as np
import seaborn as sns

### this are the files we get from kaggle:

In [5]:
import os
print(os.listdir(path))

['rating.csv', 'genome_tags.csv', 'link.csv', 'tag.csv', 'genome_scores.csv', 'movie.csv']


### rating

In [ ]:
ratings = pd.read_csv(os.path.join(path, 'rating.csv'), nrows=1_000_000
)
ratings = ratings.drop(columns=['timestamp'])
ratings.head(5)

,userId,movieId,rating
0,1,2,3.5
1,1,29,3.5
2,1,32,3.5
3,1,47,3.5
4,1,50,3.5


### movies 

In [7]:
movies = pd.read_csv(os.path.join(path, 'movie.csv'))
movies.head(5)

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


## Normalizations

### Encode genres

In [8]:
from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer()
if "genres" in movies.columns:
    genre_features = mlb.fit_transform(movies['genres'].str.split('|'))
    genre_df = pd.DataFrame(
        genre_features,
        columns=mlb.classes_,
        index=movies.index
    )

    movies = pd.concat(
        [movies[["movieId"]], genre_df],
        axis=1
    )
movies.head(5)

,movieId,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,0,0,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,0,0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,0,0,0,0,0,1,0,0,1,...,0,0,0,0,0,1,0,0,0,0
4,5,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Learing

now we will suggest a learning algorithm for this problem

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    ratings[["userId", "movieId"]],
    ratings["rating"],
    test_size=0.2,
    random_state=42
)

## LinerRegression

In [10]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)


,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


### score

In [11]:
def print_error(model, X_train, y_train, X_test, y_test):
    from sklearn.metrics import mean_absolute_error, mean_squared_error

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_mae = mean_absolute_error(y_train, train_pred)
    test_mae = mean_absolute_error(y_test, test_pred)

    train_rmse = mean_squared_error(y_train, train_pred) ** 0.5
    test_rmse = mean_squared_error(y_test, test_pred) ** 0.5

    print("Train MAE:", train_mae)
    print("Test MAE:", test_mae)
    print("Train RMSE:", train_rmse)
    print("Test RMSE:", test_rmse)
    
print_error(model, X_train, y_train, X_test, y_test)

Train MAE: 0.8546853210321494
Test MAE: 0.8481805364470141
Train RMSE: 1.0648756570269287
Test RMSE: 1.0549652559236256


As we can see we get a bad RMSE so we need to choose another model class

## Matrix Factorization
